In [1]:
import pandas as pd
import matplotlib.pyplot as plt


plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 1 график (сетка и цветные карты расстояний) 

data_grid = pd.read_csv('grid_slice.csv')


size = data_grid['i'].max()
X = data_grid['x'].values.reshape(size, size)
Y = data_grid['y'].values.reshape(size, size)
DX = data_grid['dx'].values.reshape(size, size)
DXY = data_grid['dxy'].values.reshape(size, size)


fig, (pic1, pic2, pic3) = plt.subplots(1, 3, figsize=(15, 4.5))


pic1.plot(X, Y, color='black', linewidth=0.7)
pic1.plot(X.T, Y.T, color='black', linewidth=0.7)
pic1.set_title('Исходная сетка (срез XOY)')
pic1.set_xlabel('x')
pic1.set_ylabel('y')
pic1.set_aspect('equal')


map1 = pic2.pcolormesh(X, Y, DX, shading='auto', cmap='viridis')
plt.colorbar(map1, ax=pic2)
pic2.set_title('Расстояние d^(x)')
pic2.set_xlabel('x')
pic2.set_aspect('equal')


map2 = pic3.pcolormesh(X, Y, DXY, shading='auto', cmap='plasma')
plt.colorbar(map2, ax=pic3)
pic3.set_title('Расстояние d^(xy)')
pic3.set_xlabel('x')
pic3.set_aspect('equal')

plt.tight_layout()
plt.savefig('1_grid_and_distances.png', dpi=300)
plt.close()


# 2 график влияние порядка циклов и оптимизации -O0 / -O2 

data_p2 = pd.read_csv('res_p2_loops.csv')

methods = ['SERIAL', 'serial_jki', 'serial_ijk']
time_O0 = [data_p2[(data_p2['opt'] == 'O0') & (data_p2['method'] == m)]['time'].values[0] for m in methods]
time_O2 = [data_p2[(data_p2['opt'] == 'O2') & (data_p2['method'] == m)]['time'].values[0] for m in methods]

x_pos = [0, 1, 2]
width = 0.35

plt.figure(figsize=(8, 5))
bars1 = plt.bar([p - width/2 for p in x_pos], time_O0, width=width, label='Без оптимизации (-O0)', color='coral')
bars2 = plt.bar([p + width/2 for p in x_pos], time_O2, width=width, label='С оптимизацией (-O2)', color='seagreen')

plt.bar_label(bars1, fmt='%.3f', padding=3)
plt.bar_label(bars2, fmt='%.3f', padding=3)

plt.xticks(x_pos, ['SERIAL (k-j-i)', 'serial_jki (j-k-i)', 'serial_ijk (i-j-k)'])
plt.ylabel('Время работы (секунды)')
plt.title('Сравнение порядка циклов и флагов компиляции')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig('2_loops_optimization.png', dpi=300)
plt.close()

# 3 график куда ставить !$omp do и выбор OMP_SCHEDULE 

data_p5 = pd.read_csv('res_p5_placement.csv')
data_p6 = pd.read_csv('res_p6_schedule.csv')

fig, (pic1, pic2) = plt.subplots(1, 2, figsize=(14, 5))

# Слева: положение директивы (внешний, средний или внутренний цикл)
b1 = pic1.bar(data_p5['method'], data_p5['time'], color=['steelblue', 'skyblue', 'tomato'])
pic1.bar_label(b1, fmt='%.4f', padding=3)
pic1.set_title('Место директивы !$omp do (4 потока)')
pic1.set_ylabel('Время работы (секунды)')
pic1.grid(axis='y', linestyle='--', alpha=0.5)

# Справа: разные режимы распределения задач (OMP_SCHEDULE)
b2 = pic2.bar(data_p6['schedule'], data_p6['time'], color='royalblue')
pic2.bar_label(b2, fmt='%.4f', padding=3, fontsize=9)
pic2.set_title('Режимы планировщика OMP_SCHEDULE (4 потока)')
pic2.set_ylabel('Время работы (секунды)')
pic2.tick_params(axis='x', rotation=25)
pic2.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('3_placement_and_schedule.png', dpi=300)
plt.close()

# 4график объединение циклов collapse(1, 2, 3) 

data_p7 = pd.read_csv('res_p7_collapse.csv')

fig, (pic1, pic2) = plt.subplots(1, 2, figsize=(14, 5))
chunks = [1, 10, 100]
x_pos = [0, 1, 2]
w = 0.25

for pic, opt_flag in [(pic1, 'O0'), (pic2, 'O2')]:
    sub = data_p7[data_p7['opt'] == opt_flag]
    
    t_col1 = [sub[(sub['collapse_m'] == 1) & (sub['chunk'] == c)]['time'].values[0] for c in chunks]
    t_col2 = [sub[(sub['collapse_m'] == 2) & (sub['chunk'] == c)]['time'].values[0] for c in chunks]
    t_col3 = [sub[(sub['collapse_m'] == 3) & (sub['chunk'] == c)]['time'].values[0] for c in chunks]
    
    r1 = pic.bar([p - w for p in x_pos], t_col1, width=w, label='collapse(1)')
    r2 = pic.bar(x_pos, t_col2, width=w, label='collapse(2)')
    r3 = pic.bar([p + w for p in x_pos], t_col3, width=w, label='collapse(3)')
    
    pic.bar_label(r1, fmt='%.3f', padding=2, fontsize=8)
    pic.bar_label(r2, fmt='%.3f', padding=2, fontsize=8)
    pic.bar_label(r3, fmt='%.3f', padding=2, fontsize=8)
    
    pic.set_xticks(x_pos)
    pic.set_xticklabels(['блок = 1', 'блок = 10', 'блок = 100'])
    pic.set_title(f'Работа collapse(m) с флагом -{opt_flag}')
    pic.set_ylabel('Время работы (секунды)')
    pic.grid(axis='y', linestyle='--', alpha=0.5)
    pic.legend()

plt.tight_layout()
plt.savefig('4_collapse_study.png', dpi=300)
plt.close()

# 5 график ускорение и эффективность на 1..8 потоках (Пункт 8)

data_p8 = pd.read_csv('res_p8_scaling.csv')


T1 = data_p8[data_p8['method'] == 'SERIAL']['time'].values[0]


df_loop = data_p8[data_p8['method'] == 'omp_outer'].sort_values('threads')
df_sec  = data_p8[data_p8['method'] == 'SECTIONS'].sort_values('threads')

threads = df_loop['threads'].values


S_loop = T1 / df_loop['time'].values
E_loop = S_loop / threads

S_sec = T1 / df_sec['time'].values
E_sec = S_sec / threads

fig, (pic1, pic2) = plt.subplots(1, 2, figsize=(14, 5))


pic1.plot(threads, threads, 'k--', label='Идеальное ускорение')
pic1.plot(threads, S_loop, 'o-', color='seagreen', linewidth=2, label='Параллельный цикл (!$omp do)')
pic1.plot(threads, S_sec, 's-', color='darkorange', linewidth=2, label='Параллельные секции (!$omp sections)')
pic1.set_title('Ускорение программы (Sp)')
pic1.set_xlabel('Число потоков (p)')
pic1.set_ylabel('Ускорение (во сколько раз быстрее)')
pic1.set_xticks(threads)
pic1.grid(True, linestyle='--', alpha=0.5)
pic1.legend()

pic2.plot(threads, [1.0]*len(threads), 'k--', label='Идеальная эффективность (100%)')
pic2.plot(threads, E_loop, 'o-', color='seagreen', linewidth=2, label='Параллельный цикл (!$omp do)')
pic2.plot(threads, E_sec, 's-', color='darkorange', linewidth=2, label='Параллельные секции (!$omp sections)')
pic2.set_title('Эффективность распараллеливания (Ep)')
pic2.set_xlabel('Число потоков (p)')
pic2.set_ylabel('Эффективность (от 0 до 1)')
pic2.set_xticks(threads)
pic2.grid(True, linestyle='--', alpha=0.5)
pic2.legend()

plt.tight_layout()
plt.savefig('5_speedup_and_efficiency.png', dpi=300)
plt.close()

print("Все 5 картинок успешно сохранены в папку!")


Все 5 картинок успешно сохранены в папку!
